# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khalilzufar/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row = One unique URL at a specific snapshot date ($T_0$).Time Windows:
- Historical Feature Window ($T_0$): 30 days of historical Google Search Console performance data (clicks, impressions, position) aggregated up to the snapshot date $T_0$.
- Observed Label Window ($T_1$): The subsequent 30 days immediately following $T_0$ to observe actual performance outcome (whether traffic dropped by $>20\%$).
- Data Slice Used: Mid-panel month (e.g., month=2026-03) to avoid boundary overlap and testing data leakage.

In [5]:
import pandas as pd
import numpy as np

# Verify unit of analysis setup in code
print("Unit of Analysis: 1 Row = 1 Unique URL at Snapshot Date T0")
print("Feature Window (T0): [T0 - 30 days, T0]")
print("Label Observation Window (T1): (T0, T0 + 30 days]")

Unit of Analysis: 1 Row = 1 Unique URL at Snapshot Date T0
Feature Window (T0): [T0 - 30 days, T0]
Label Observation Window (T1): (T0, T0 + 30 days]


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Field Bucket Sorting:
- Feature Fields (Aggregated at $T_0$):
  - total_clicks_30d (Numerical) — Total organic clicks in historical window.
  - total_impressions_30d (Numerical) — Total search impressions in historical window.
  - avg_ctr_30d (Numerical) — Historical Click-Through Rate.
  - mean_position_30d (Numerical) — Average search ranking position.
  - position_drift_14d (Numerical) — Short-term ranking position shift.
- Label Field (Observed at $T_1$):
  - is_declining_next_30d (Binary: 1 if clicks in $T_1$ drop $>20\%$ vs $T_0$, else 0).
- Context Fields (Metadata/Identifiers):
  - url (String) — Page unique identifier.
    - snapshot_date (Date) — Date of evaluation ($T_0$).
- Excluded Fields & Reason:
  - raw_logs, session_id, future_clicks_t1 — Excluded to prevent data leakage and eliminate non-informative high-cardinality noise.

In [6]:
# Create data contract schema definition dictionary
data_contract_schema = {
    'features': ['total_clicks_30d', 'total_impressions_30d', 'avg_ctr_30d', 'mean_position_30d', 'position_drift_14d'],
    'label': ['is_declining_next_30d'],
    'context': ['url', 'snapshot_date'],
    'excluded': ['future_clicks_t1', 'session_id', 'raw_logs']
}

for bucket, fields in data_contract_schema.items():
    print(f"Bucket [{bucket.upper()}]: {fields}")

Bucket [FEATURES]: ['total_clicks_30d', 'total_impressions_30d', 'avg_ctr_30d', 'mean_position_30d', 'position_drift_14d']
Bucket [LABEL]: ['is_declining_next_30d']
Bucket [CONTEXT]: ['url', 'snapshot_date']
Bucket [EXCLUDED]: ['future_clicks_t1', 'session_id', 'raw_logs']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Contract Verification Queries:
1. Grain Check: Confirming that (url, snapshot_date) is unique and contains no duplicate records.
2. Count & Missing Value Audit: Verifying total row count and checking zero/null rate in key features.
3. Window Boundary Verification: Ensuring feature aggregation strictly ends at $T_0$ without peeking into $T_1$.

In [7]:
# Generate synthetic dataset to execute contract verification queries
np.random.seed(42)
dates = pd.date_range(start="2026-03-01", end="2026-03-31")
urls = [f"/blog/article-{i}" for i in range(1, 101)]

rows = []
for d in dates:
    for u in urls:
        clicks_t0 = np.random.poisson(lam=20)
        impressions_t0 = clicks_t0 * np.random.randint(10, 25)
        # Simulate observed future clicks in T1
        clicks_t1 = int(clicks_t0 * np.random.uniform(0.6, 1.2))

        rows.append({
            'snapshot_date': d,
            'url': u,
            'clicks_t0': clicks_t0,
            'impressions_t0': impressions_t0,
            'avg_position_t0': np.random.uniform(1.0, 15.0),
            'clicks_t1_observed': clicks_t1
        })

df_raw = pd.DataFrame(rows)

# Query 1: Verify Grain Uniqueness
duplicates = df_raw.duplicated(subset=['url', 'snapshot_date']).sum()
print(f"[Query 1 - Grain Check] Duplicate (URL, Snapshot) rows: {duplicates}")

# Query 2: Missing Values & Row Counts
print(f"[Query 2 - Volume Check] Total Rows: {len(df_raw)} | Total Unique URLs: {df_raw['url'].nunique()}")
print(f"[Query 2 - Missing Check] Null values in dataset:\n{df_raw.isnull().sum()}")

# Query 3: Calculate Target and Verify Feature Window
df_raw['is_declining_next_30d'] = ((df_raw['clicks_t1_observed'] - df_raw['clicks_t0']) / df_raw['clicks_t0'] < -0.20).astype(int)
print(f"[Query 3 - Target Balance] Target distribution:\n{df_raw['is_declining_next_30d'].value_counts(normalize=True)}")

[Query 1 - Grain Check] Duplicate (URL, Snapshot) rows: 0
[Query 2 - Volume Check] Total Rows: 3100 | Total Unique URLs: 100
[Query 2 - Missing Check] Null values in dataset:
snapshot_date         0
url                   0
clicks_t0             0
impressions_t0        0
avg_position_t0       0
clicks_t1_observed    0
dtype: int64
[Query 3 - Target Balance] Target distribution:
is_declining_next_30d
0    0.63
1    0.37
Name: proportion, dtype: float64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Inherent Data Limitations:

1. Unbalanced History / New Pages: Brand new URLs without 30 days of historical data cannot produce reliable rolling metrics.

2. External Google Algorithm Updates: Search Console metrics reflect external Google algorithm changes, which are unobserved in raw click logs and create sudden distribution shifts.

3. Window Overlap Correlation: Consecutive snapshot dates (e.g., March 1 vs March 2) share 29 days of overlapping historical data, introducing strong temporal autocorrelation that requires careful validation splitting (time-based split rather than random split).

In [8]:
# Check data limitation: identify new URLs with low historical data threshold
low_data_pages = df_raw[df_raw['impressions_t0'] < 10]
print(f"[Data Limit Audit] Rows with insufficient historical search exposure (<10 impressions): {len(low_data_pages)}")
print("Limitation Note: Time-based train/test splits must be used due to 29-day rolling window overlap.")

[Data Limit Audit] Rows with insufficient historical search exposure (<10 impressions): 0
Limitation Note: Time-based train/test splits must be used due to 29-day rolling window overlap.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅]  The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅]  No client names, URLs, or private queries anywhere
- [✅]  My claims use careful words: observed, measured, directional, decision-support
- [✅]  Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.